In [ ]:
!pip install chromadb sentence-transformers groq pandas --q

In [ ]:
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer
from groq import Groq
import os

print("All libraries imported successfully!")
print("Ready to build RAG system")

All libraries imported successfully!
Ready to build RAG system


In [ ]:
GROQ_API_KEY=""
os.environ["GROQ_API_KEY"]=GROQ_API_KEY
groq_client=Groq(api_key=GROQ_API_KEY)

print("Groq API client initialized successfully!")

Groq API client initialized successfully!


In [ ]:
df=pd.read_csv('college_notes.csv')
print("Shape of Dataset:",df.shape)
print("\ncolumn names:",df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))
print(df.tail(3))

Shape of Dataset: (15, 4)

column names: ['note_id', 'subject', 'topic', 'content']

First 3 rows:
   note_id           subject             topic  \
0        1  Data Engineering     ETL Pipelines   
1        2  Data Engineering  Data Warehousing   
2        3  Data Engineering      Apache Spark   

                                             content  
0  ETL stands for Extract, Transform, Load. It is...  
1  A data warehouse is a central repository that ...  
2  Apache Spark is an open-source distributed com...  
    note_id subject            topic  \
12       13   GenAI      RAG Systems   
13       14  Python   Pandas Library   
14       15  Python  API Integration   

                                              content  
12  Retrieval-Augmented Generation or RAG is an AI...  
13  Pandas is a Python library for data manipulati...  
14  An API or Application Programming Interface al...  


In [ ]:
print("Subject in dataset")
print(df['subject'].value_counts())
print("\nSample of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nLength of contetn(number of characters)for each note:")
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

Subject in dataset
subject
Data Engineering    5
GenAI               5
Machine Learning    3
Python              2
Name: count, dtype: int64

Sample of topics:
 note_id          subject                  topic
       1 Data Engineering          ETL Pipelines
       2 Data Engineering       Data Warehousing
       3 Data Engineering           Apache Spark
       4 Data Engineering Medallion Architecture
       5 Data Engineering         Data Pipelines
       6 Machine Learning      Linear Regression
       7 Machine Learning    Feature Engineering
       8 Machine Learning       Model Evaluation
       9            GenAI  Large Language Models
      10            GenAI     Prompt Engineering
      11            GenAI             Embeddings
      12            GenAI       Vector Databases
      13            GenAI            RAG Systems
      14           Python         Pandas Library
      15           Python        API Integration

Length of contetn(number of characters)for each note:
 

In [ ]:
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}"for row in df.to_dict('records')]
metadatas=[
    {"subject": row['subject'],"topic": row['topic']}
    for row in df.to_dict('records')
]
print(f"total chunks prepared: {len(documents)}")
print(f"First document ID : {ids[14]}")
print(f"First document metadata : {metadatas[14]}")
print(f"First 100 chars of doc: {documents[14][:100]}...")

total chunks prepared: 15
First document ID : note_15
First document metadata : {'subject': 'Python', 'topic': 'API Integration'}
First 100 chars of doc: An API or Application Programming Interface allows different software systems to communicate with ea...


In [ ]:
embedding_model=SentenceTransformer('all-miniLM-L6-v2')
print("\nEmbedding model loaded successfully")
test_embedding= embedding_model.encode("This is a test sentence")
print(f"Test embedding shape: {test_embedding.shape}")
print(f"First 5 values of a test embedding: {test_embedding[:5]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-miniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding model loaded successfully
Test embedding shape: (384,)
First 5 values of a test embedding: [0.07155243 0.06848023 0.00660337 0.10176966 0.01112225]


In [ ]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection(name="college_notes_rag")
print("ChromaDB client created")
print(f"Collection name: college_notes_rag")
print(f"Documents in collection so far: {collection.count()}")


ChromaDB client created
Collection name: college_notes_rag
Documents in collection so far: 0


In [ ]:
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print(f"\nEmbedding matrix shape: {embeddings.shape}")
embeddings_list=embeddings.tolist()

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings_list
)
print(f"\nDocumments successfully addedd to collection")
print(f"Documents in collection now: {collection.count()}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (15, 384)

Documments successfully addedd to collection
Documents in collection now: 15
